# Severstal Steel Defect Detection 

## Fast review

Run the quick summary cell below if you only want to review the saved results. 



In [ ]:
import os
import gc
import json
import time
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
)
import segmentation_models_pytorch as smp
import albumentations as A


In [ ]:
# Quick saved-summary view. Run this cell only if you do not want to train/evaluate.
import os

summary_path = os.path.join("outputs_improved_strongest", "final_project_summary.txt")

if os.path.exists(summary_path):
    with open(summary_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print(f"Saved summary not found: {summary_path}")


In [ ]:
# Quick saved-visuals view. Run this cell only if you do not want to train/evaluate.
import glob
import os

import matplotlib.pyplot as plt
from IPython.display import display

visual_paths = sorted(glob.glob(os.path.join("outputs_improved_strongest", "visuals", "*.png")))

if not visual_paths:
    print("No saved visuals found in outputs_improved_strongest/visuals")
else:
    cols = 2
    rows = (len(visual_paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(14, 5 * rows))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

    for ax, path in zip(axes, visual_paths):
        img = plt.imread(path)
        ax.imshow(img)
        ax.set_title(os.path.basename(path), fontsize=10)
        ax.axis("off")

    for ax in axes[len(visual_paths):]:
        ax.axis("off")

    plt.tight_layout()
    display(fig)
    plt.close(fig)


In [ ]:
# Configuration
SEED = 42

BASE_PATH = os.environ.get(
    "SEVERSTAL_BASE_PATH",
    os.path.expanduser("~/Downloads/severstal-steel-defect-detection"),
)
TRAIN_PATH = os.path.join(BASE_PATH, "train_images")
CSV_PATH = os.path.join(BASE_PATH, "train.csv")

OUT_DIR = "outputs_improved_strongest"
VIS_DIR = os.path.join(OUT_DIR, "visuals")
CKPT_DIR = os.path.join(OUT_DIR, "checkpoints")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# `resnet34` is a stable default for limited VRAM.
ENCODER_NAME = "resnet34"
ENCODER_WEIGHTS = "imagenet"

BATCH = 6
EPOCHS_CLS = 10
EPOCHS_SEG = 40
EPOCHS_REF = 20

LR = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0

# Keep the classifier threshold low so true defects are not filtered out early.
TH_CLS = 0.10

TH_SEG = 0.30
TH_REF = 0.30
TH_ROI = 0.10

ROI_PAD = 260
DILATE_K = 21
DILATE_I = 3

FINAL_DIL_K = 5
FINAL_DIL_I = 1

CROP_SIZE = 512

USE_TTA_INFERENCE = True
USE_TTA_THRESHOLD_SEARCH = False
USE_CONNECTED_COMPONENT_FILTER = True
MIN_COMPONENT_AREA = 20

# Notebook runs are more stable on macOS with DataLoader workers disabled.
NUM_WORKERS = 0
CPU_THREADS = max(1, (os.cpu_count() or 4) - 2)

# Resume uses the last saved checkpoint when available.
RESUME = True

FORCE_RETRAIN = False


In [ ]:
# Runtime setup
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.set_num_threads(CPU_THREADS)
try:
    torch.set_num_interop_threads(max(1, min(4, CPU_THREADS)))
except RuntimeError:
    pass

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

def choose_device():
    if torch.cuda.is_available():
        return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


device = choose_device()
scaler = GradScaler("cuda") if device == "cuda" else None

print("Device:", device)
if device == "cpu" and hasattr(torch.backends, "mps"):
    print("MPS built:", torch.backends.mps.is_built())
    print("MPS available:", torch.backends.mps.is_available())
print("Encoder:", ENCODER_NAME)
print("Batch size:", BATCH)
print("DataLoader workers:", NUM_WORKERS)
print("Torch CPU threads:", torch.get_num_threads())
print("Torch interop threads:", torch.get_num_interop_threads())

loader_kwargs = {
    "num_workers": NUM_WORKERS,
    "pin_memory": device == "cuda",
}
if NUM_WORKERS > 0:
    loader_kwargs.update({
        "persistent_workers": True,
        "prefetch_factor": 2,
    })

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")
if not os.path.isdir(TRAIN_PATH):
    raise FileNotFoundError(f"Images folder not found: {TRAIN_PATH}")

df = pd.read_csv(CSV_PATH)


In [ ]:
# Data augmentation
seg_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.GaussNoise(var_limit=(5, 30), p=0.3),
    A.Blur(blur_limit=3, p=0.2),
    A.ShiftScaleRotate(
        shift_limit=0.02,
        scale_limit=0.1,
        rotate_limit=3,
        border_mode=cv2.BORDER_REFLECT,
        p=0.4,
    ),
    A.GridDistortion(p=0.2),
    A.CLAHE(clip_limit=2.0, p=0.3),
])

cls_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.4),
    A.GaussNoise(p=0.3),
    A.CLAHE(p=0.3),
])

crop_aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    A.CLAHE(p=0.3),
    # Crop-level deformation helps the refiner handle local shape variation.
    A.ElasticTransform(alpha=1, sigma=10, p=0.15),
])


In [ ]:
# Utility functions
def safe_cache():
    if device == "cuda":
        torch.cuda.empty_cache()
    elif device == "mps" and hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()
    gc.collect()


def read_u8(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Image unreadable or missing: {path}")
    return img


def read_f32(path):
    return read_u8(path).astype(np.float32) / 255.0


def rle_decode(mask_rle, shape=(256, 1600)):
    if pd.isna(mask_rle):
        return np.zeros(shape, dtype=np.uint8)

    s = list(map(int, mask_rle.split()))
    starts = np.array(s[0::2]) - 1
    lengths = np.array(s[1::2])
    ends = starts + lengths

    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1

    return img.reshape((shape[1], shape[0])).T


def build_mask(dataframe, image_id):
    mask = np.zeros((256, 1600), dtype=np.uint8)
    rows = dataframe[dataframe["ImageId"] == image_id]

    for _, row in rows.iterrows():
        if not pd.isna(row["EncodedPixels"]):
            mask = np.maximum(mask, rle_decode(row["EncodedPixels"]))

    return mask


def has_defect(dataframe, image_id):
    return int(dataframe[dataframe["ImageId"] == image_id]["EncodedPixels"].notna().any())


def get_bbox(mask, pad=220):
    ys, xs = np.where(mask > 0)

    if len(xs) == 0 or len(ys) == 0:
        return None

    return (
        max(0, xs.min() - pad),
        max(0, ys.min() - pad),
        min(1599, xs.max() + pad),
        min(255, ys.max() + pad),
    )


def to_tensor(img_f):
    return torch.from_numpy(
        np.stack([img_f, img_f, img_f], axis=0)
    ).float().unsqueeze(0).to(device)


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=4)


def checkpoint_path(name):
    return os.path.join(CKPT_DIR, name)


def save_checkpoint(path, model, optimizer, scheduler, epoch, best_metric, history):
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "best_metric": best_metric,
        "history": history,
        "encoder": ENCODER_NAME,
    }, path)


def load_checkpoint_if_exists(path, model, optimizer=None, scheduler=None):
    if not os.path.exists(path):
        return 0, -1.0, []

    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state"])

    if optimizer is not None and ckpt.get("optimizer_state") is not None:
        optimizer.load_state_dict(ckpt["optimizer_state"])

    if scheduler is not None and ckpt.get("scheduler_state") is not None:
        scheduler.load_state_dict(ckpt["scheduler_state"])

    start_epoch = int(ckpt.get("epoch", 0))
    best_metric = float(ckpt.get("best_metric", -1.0))
    history = ckpt.get("history", [])

    print(f"  Resumed from {path} at epoch {start_epoch}")
    return start_epoch, best_metric, history


def find_checkpoint(name):
    stem, ext = os.path.splitext(name)
    candidates = [name]
    if ext == "":
        candidates.extend([f"{name}.pth", f"{name}.pt", f"{name}.ckpt"])

    for candidate in candidates:
        path = checkpoint_path(candidate)
        if os.path.exists(path):
            return path

    return None


def load_model_weights(path, model, strict=True):
    ckpt = torch.load(path, map_location=device)

    if isinstance(ckpt, dict) and "model_state" in ckpt:
        state = ckpt["model_state"]
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state = ckpt["state_dict"]
    else:
        state = ckpt

    if isinstance(state, dict):
        state = {k.replace("module.", "", 1): v for k, v in state.items()}

    model.load_state_dict(state, strict=strict)
    print(f"Loaded weights: {path}")
    return model


def skipped_metrics(stage_name, path):
    return {
        "stage": stage_name,
        "loaded_from_checkpoint": path,
        "training_skipped_this_run": True,
        "training_time_seconds_this_run": 0.0,
        "training_time_minutes": 0.0,
    }


def load_saved_threshold():
    path = os.path.join(OUT_DIR, "best_threshold.json")
    if not os.path.exists(path):
        return None

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if "best_threshold" not in data:
        return None

    best_th = float(data["best_threshold"])
    best_dice = float(data.get("best_val_dice", -1.0))
    used_tta = bool(data.get("used_tta", False))
    print(
        f"Loaded saved threshold: {best_th:.4f} | "
        f"saved val Dice: {best_dice:.4f} | used_tta: {used_tta}"
    )
    return best_th, best_dice, used_tta


def filter_small_components(mask, min_area=20):
    if not USE_CONNECTED_COMPONENT_FILTER:
        return mask.astype(np.uint8)

    mask = mask.astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)

    cleaned = np.zeros_like(mask, dtype=np.uint8)
    for label in range(1, num_labels):
        area = stats[label, cv2.CC_STAT_AREA]
        if area >= min_area:
            cleaned[labels == label] = 1

    return cleaned


In [ ]:
# Metrics
def dice_t(logits, target, th=0.30):
    pred = (torch.sigmoid(logits) > th).float()
    inter = (pred * target).sum()
    return (2.0 * inter + 1e-6) / (pred.sum() + target.sum() + 1e-6)


def dice_np(pred, target, smooth=1e-6):
    pred = pred.astype(np.uint8)
    target = target.astype(np.uint8)

    if pred.sum() == 0 and target.sum() == 0:
        return 1.0

    inter = (pred * target).sum()
    return (2.0 * inter + smooth) / (pred.sum() + target.sum() + smooth)


def iou_np(pred, target, smooth=1e-6):
    pred = pred.astype(np.uint8)
    target = target.astype(np.uint8)

    if pred.sum() == 0 and target.sum() == 0:
        return 1.0

    inter = (pred * target).sum()
    union = pred.sum() + target.sum() - inter

    return (inter + smooth) / (union + smooth)


In [ ]:
# Loss functions
dice_loss = smp.losses.DiceLoss(mode="binary", smooth=1.0)
tversky_loss = smp.losses.TverskyLoss(mode="binary", alpha=0.3, beta=0.7)


class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, target):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits,
            target,
            reduction="none",
        )
        prob = torch.sigmoid(logits)
        pt = prob * target + (1.0 - prob) * (1.0 - target)
        alpha_t = self.alpha * target + (1.0 - self.alpha) * (1.0 - target)
        loss = alpha_t * (1.0 - pt).pow(self.gamma) * bce
        return loss.mean()


focal_loss = BinaryFocalLoss(alpha=0.75, gamma=2.0)


def seg_loss(pred, target):
    return (
        0.4 * dice_loss(pred, target)
        + 0.4 * focal_loss(pred, target)
        + 0.2 * tversky_loss(pred, target)
    )


In [ ]:
# Train, validation, and test split
all_ids = sorted([f for f in os.listdir(TRAIN_PATH) if f.endswith(".jpg")])
if len(all_ids) == 0:
    raise RuntimeError(f"No .jpg images found in {TRAIN_PATH}")

labels = [has_defect(df, i) for i in all_ids]

train_val_ids, test_ids = train_test_split(
    all_ids,
    test_size=0.20,
    random_state=SEED,
    stratify=labels,
)

train_val_labels = [has_defect(df, i) for i in train_val_ids]

train_ids, val_ids = train_test_split(
    train_val_ids,
    test_size=0.25,
    random_state=SEED,
    stratify=train_val_labels,
)

print(f"Train: {len(train_ids)} | Val: {len(val_ids)} | Test: {len(test_ids)}")

save_json({
    "train": len(train_ids),
    "val": len(val_ids),
    "test": len(test_ids),
    "seed": SEED,
    "batch": BATCH,
    "encoder": ENCODER_NAME,
}, os.path.join(OUT_DIR, "split_info.json"))


In [ ]:
# Dataset classes
class ClsDataset(Dataset):
    def __init__(self, ids, aug=None):
        self.ids = list(ids)
        self.aug = aug
        self.labels = [has_defect(df, i) for i in self.ids]

        pos = sum(self.labels)
        neg = len(self.labels) - pos
        print(f"  Classifier dataset | pos: {pos} | neg: {neg}")

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        img = read_u8(os.path.join(TRAIN_PATH, image_id))

        if self.aug:
            img = self.aug(image=img)["image"]

        img = img.astype(np.float32) / 255.0
        img = np.stack([img, img, img], axis=0)

        label = self.labels[idx]
        return torch.from_numpy(img).float(), torch.tensor([label], dtype=torch.float32)


class SegDataset(Dataset):
    def __init__(self, ids, aug=None):
        self.ids = list(ids)
        self.aug = aug

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image_id = self.ids[idx]

        img = read_u8(os.path.join(TRAIN_PATH, image_id))
        mask = build_mask(df, image_id)

        if self.aug:
            out = self.aug(image=img, mask=mask)
            img = out["image"]
            mask = out["mask"]

        img = img.astype(np.float32) / 255.0
        img = np.stack([img, img, img], axis=0)

        mask = mask[None].astype(np.float32)

        return torch.from_numpy(img).float(), torch.from_numpy(mask).float()


class CropDataset(Dataset):
    """
    Stage 2 is trained from Stage 1 predictions.
    Stage 1 learns the whole-image context; Stage 2 refines the predicted ROI.
    """
    def __init__(self, ids, seg_model, aug=None):
        self.ids = list(ids)
        self.aug = aug
        self.samples = []

        print("  Building crop dataset from Stage 1 predictions...")
        seg_model.eval()

        for image_id in tqdm(self.ids, desc="  Building crops"):
            img_f = read_f32(os.path.join(TRAIN_PATH, image_id))
            tensor = to_tensor(img_f)

            with torch.no_grad():
                prob = torch.sigmoid(seg_model(tensor))[0, 0].cpu().numpy()

            roi = (prob > TH_ROI).astype(np.uint8)
            roi = cv2.dilate(
                roi,
                np.ones((DILATE_K, DILATE_K), np.uint8),
                iterations=DILATE_I,
            )

            bbox = get_bbox(roi, pad=ROI_PAD)

            if bbox is not None:
                x1, y1, x2, y2 = bbox
                gt = build_mask(df, image_id)

                if gt[y1:y2+1, x1:x2+1].sum() > 0 or random.random() < 0.15:
                    self.samples.append((image_id, x1, y1, x2, y2))

            del tensor

        safe_cache()
        print(f"  Crop samples: {len(self.samples)}")

        if len(self.samples) == 0:
            raise RuntimeError(
                "CropDataset is empty. Lower TH_ROI, increase ROI_PAD, or train Stage 1 longer."
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_id, x1, y1, x2, y2 = self.samples[idx]

        img = read_u8(os.path.join(TRAIN_PATH, image_id))
        mask = build_mask(df, image_id)

        crop_img = img[y1:y2+1, x1:x2+1]
        crop_mask = mask[y1:y2+1, x1:x2+1]

        if crop_img.size == 0 or crop_mask.size == 0:
            raise RuntimeError(f"Empty crop created for image: {image_id}")

        if self.aug:
            out = self.aug(image=crop_img, mask=crop_mask)
            crop_img = out["image"]
            crop_mask = out["mask"]

        crop_img = cv2.resize(crop_img, (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_LINEAR)
        crop_mask = cv2.resize(crop_mask, (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_NEAREST)

        crop_img = crop_img.astype(np.float32) / 255.0
        crop_img = np.stack([crop_img, crop_img, crop_img], axis=0)
        crop_mask = crop_mask[None].astype(np.float32)

        return torch.from_numpy(crop_img).float(), torch.from_numpy(crop_mask).float()


In [ ]:
# Model definitions
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = smp.encoders.get_encoder(
            name=ENCODER_NAME,
            in_channels=3,
            depth=5,
            weights=ENCODER_WEIGHTS,
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(0.3)

        out_channels = self.encoder.out_channels[-1]
        self.fc = nn.Linear(out_channels, 1)

    def forward(self, x):
        features = self.encoder(x)
        x = features[-1]
        x = self.pool(x).flatten(1)
        x = self.drop(x)
        return self.fc(x)


def make_fpn():
    return smp.FPN(
        encoder_name=ENCODER_NAME,
        encoder_weights=ENCODER_WEIGHTS,
        in_channels=3,
        classes=1,
        decoder_pyramid_channels=256,
        decoder_segmentation_channels=128,
    ).to(device)


In [ ]:
# Training utilities
def make_optimizer(model):
    return torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )


def make_scheduler(optimizer):
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    )


def train_step(model, loss_fn, optimizer, images, targets):
    optimizer.zero_grad(set_to_none=True)

    if scaler is not None:
        with autocast(device_type="cuda"):
            preds = model(images)
            loss = loss_fn(preds, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
    else:
        preds = model(images)
        loss = loss_fn(preds, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()

    return loss.item()


In [ ]:
# Stage 0: image-level classifier
def train_classifier(resume=True):
    print("\n" + "=" * 60)
    print("STAGE 0 — CLASSIFIER")
    print("=" * 60)

    start_time = time.perf_counter()

    model = Classifier().to(device)

    pos = sum(has_defect(df, i) for i in train_ids)
    neg = len(train_ids) - pos

    if pos == 0:
        raise RuntimeError("No positive samples found in training split.")

    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([neg / pos], device=device)
    )

    optimizer = make_optimizer(model)
    scheduler = make_scheduler(optimizer)

    start_epoch = 0
    best_f1 = -1.0
    history = []

    if resume:
        start_epoch, best_f1, history = load_checkpoint_if_exists(
            checkpoint_path("last_cls.pth"),
            model,
            optimizer,
            scheduler,
        )

    train_loader = DataLoader(
        ClsDataset(train_ids, aug=cls_aug),
        batch_size=BATCH,
        shuffle=True,
        **loader_kwargs,
    )

    val_loader = DataLoader(
        ClsDataset(val_ids),
        batch_size=BATCH,
        shuffle=False,
        **loader_kwargs,
    )

    print(
        f"  Train images: {len(train_loader.dataset)} | Val images: {len(val_loader.dataset)} | "
        f"Train steps/epoch: {len(train_loader)} | Val steps: {len(val_loader)}"
    )

    for epoch in range(start_epoch, EPOCHS_CLS):
        model.train()
        total_loss = 0.0

        for images, labels in tqdm(train_loader, desc=f"Cls {epoch+1}/{EPOCHS_CLS}"):
            total_loss += train_step(
                model,
                loss_fn,
                optimizer,
                images.to(device),
                labels.to(device),
            )

        model.eval()
        preds_all = []
        labels_all = []

        with torch.no_grad():
            for images, labels in val_loader:
                probs = torch.sigmoid(model(images.to(device))).cpu().numpy().ravel()
                preds = (probs > 0.5).astype(np.uint8)

                preds_all.extend(preds.tolist())
                labels_all.extend(labels.numpy().ravel().tolist())

        f1 = f1_score(labels_all, preds_all, zero_division=0)
        acc = accuracy_score(labels_all, preds_all)
        precision = precision_score(labels_all, preds_all, zero_division=0)
        recall = recall_score(labels_all, preds_all, zero_division=0)

        scheduler.step(f1)

        epoch_record = {
            "epoch": epoch + 1,
            "loss": total_loss / len(train_loader),
            "f1": f1,
            "accuracy": acc,
            "precision": precision,
            "recall": recall,
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(epoch_record)

        print(
            f"Epoch {epoch+1} | "
            f"Loss: {epoch_record['loss']:.4f} | "
            f"F1: {f1:.4f} | "
            f"Acc: {acc:.4f} | "
            f"P: {precision:.4f} | "
            f"R: {recall:.4f} | "
            f"LR: {epoch_record['lr']:.1e}"
        )

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), checkpoint_path("best_cls.pth"))
            print("Saved best_cls.pth")

        save_checkpoint(
            checkpoint_path("last_cls.pth"),
            model,
            optimizer,
            scheduler,
            epoch + 1,
            best_f1,
            history,
        )

    elapsed = time.perf_counter() - start_time
    pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "classifier_history.csv"), index=False)

    best_record = max(history, key=lambda x: x.get("f1", -1.0)) if history else None
    if best_record is not None:
        print(
            "Best classifier | "
            f"Epoch: {best_record['epoch']} | "
            f"F1: {best_record['f1']:.4f} | "
            f"Acc: {best_record['accuracy']:.4f} | "
            f"P: {best_record['precision']:.4f} | "
            f"R: {best_record['recall']:.4f}"
        )

    best_cls_path = checkpoint_path("best_cls.pth")
    if os.path.exists(best_cls_path):
        model.load_state_dict(torch.load(best_cls_path, map_location=device))
        print(f"Loaded best classifier checkpoint: {best_cls_path}")

    metrics = {
        "best_f1": best_f1,
        "best_epoch": int(best_record["epoch"]) if best_record is not None else None,
        "best_accuracy": float(best_record["accuracy"]) if best_record is not None else None,
        "best_precision": float(best_record["precision"]) if best_record is not None else None,
        "best_recall": float(best_record["recall"]) if best_record is not None else None,
        "training_time_seconds_this_run": elapsed,
        "training_time_minutes": elapsed / 60,
    }
    save_json(metrics, os.path.join(OUT_DIR, "classifier_metrics.json"))
    return model, metrics


In [ ]:
# Stage 1: full-image segmentation model
def train_segmenter(resume=True):
    print("\n" + "=" * 60)
    print("STAGE 1 — FULL IMAGE FPN")
    print("=" * 60)

    start_time = time.perf_counter()

    model = make_fpn()
    optimizer = make_optimizer(model)
    scheduler = make_scheduler(optimizer)

    start_epoch = 0
    best_dice = -1.0
    history = []

    if resume:
        start_epoch, best_dice, history = load_checkpoint_if_exists(
            checkpoint_path("last_seg.pth"),
            model,
            optimizer,
            scheduler,
        )

    train_loader = DataLoader(
        SegDataset(train_ids, aug=seg_aug),
        batch_size=BATCH,
        shuffle=True,
        **loader_kwargs,
    )

    val_loader = DataLoader(
        SegDataset(val_ids),
        batch_size=BATCH,
        shuffle=False,
        **loader_kwargs,
    )

    print(
        f"  Train images: {len(train_loader.dataset)} | Val images: {len(val_loader.dataset)} | "
        f"Train steps/epoch: {len(train_loader)} | Val steps: {len(val_loader)}"
    )

    patience = 8
    wait = 0

    for epoch in range(start_epoch, EPOCHS_SEG):
        model.train()
        total_loss = 0.0

        for step, (images, masks) in enumerate(tqdm(train_loader, desc=f"Seg {epoch+1}/{EPOCHS_SEG}")):
            loss_value = train_step(
                model,
                seg_loss,
                optimizer,
                images.to(device),
                masks.to(device),
            )
            total_loss += loss_value

        model.eval()
        dice_sum = 0.0

        with torch.no_grad():
            for images, masks in val_loader:
                logits = model(images.to(device))
                dice_sum += dice_t(logits, masks.to(device), TH_SEG).item()

        val_dice = dice_sum / len(val_loader)
        scheduler.step(val_dice)

        epoch_record = {
            "epoch": epoch + 1,
            "loss": total_loss / len(train_loader),
            "val_dice": val_dice,
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(epoch_record)

        print(
            f"Epoch {epoch+1} | "
            f"Loss: {epoch_record['loss']:.4f} | "
            f"Val Dice: {val_dice:.4f} | "
            f"LR: {epoch_record['lr']:.1e}"
        )

        improved = val_dice > best_dice
        if improved:
            best_dice = val_dice
            wait = 0
            torch.save(model.state_dict(), checkpoint_path("best_seg.pth"))
            print("Saved best_seg.pth")
        else:
            wait += 1

        save_checkpoint(
            checkpoint_path("last_seg.pth"),
            model,
            optimizer,
            scheduler,
            epoch + 1,
            best_dice,
            history,
        )

        if wait >= patience:
            print("Early stopping segmenter")
            break

    elapsed = time.perf_counter() - start_time
    pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "segmenter_history.csv"), index=False)

    metrics = {
        "best_val_dice": best_dice,
        "training_time_seconds_this_run": elapsed,
        "training_time_minutes": elapsed / 60,
    }
    save_json(metrics, os.path.join(OUT_DIR, "segmenter_metrics.json"))
    return model, metrics


In [ ]:
# Stage 2: crop-based refiner
def train_refiner(seg_model, resume=True):
    print("\n" + "=" * 60)
    print("STAGE 2 — CROP REFINER")
    print("=" * 60)

    start_time = time.perf_counter()

    model = make_fpn()
    optimizer = make_optimizer(model)
    scheduler = make_scheduler(optimizer)

    start_epoch = 0
    best_dice = -1.0
    history = []

    if resume:
        start_epoch, best_dice, history = load_checkpoint_if_exists(
            checkpoint_path("last_ref.pth"),
            model,
            optimizer,
            scheduler,
        )

    train_crop_dataset = CropDataset(train_ids, seg_model, aug=crop_aug)
    val_crop_dataset = CropDataset(val_ids, seg_model)

    save_json({
        "train_crop_samples": len(train_crop_dataset),
        "val_crop_samples": len(val_crop_dataset),
        "crop_size": CROP_SIZE,
        "th_roi": TH_ROI,
        "roi_pad": ROI_PAD,
    }, os.path.join(OUT_DIR, "crop_dataset_info.json"))

    train_loader = DataLoader(
        train_crop_dataset,
        batch_size=BATCH,
        shuffle=True,
        **loader_kwargs,
    )

    val_loader = DataLoader(
        val_crop_dataset,
        batch_size=BATCH,
        shuffle=False,
        **loader_kwargs,
    )

    print(
        f"  Train crops: {len(train_loader.dataset)} | Val crops: {len(val_loader.dataset)} | "
        f"Train steps/epoch: {len(train_loader)} | Val steps: {len(val_loader)}"
    )

    patience = 6
    wait = 0

    for epoch in range(start_epoch, EPOCHS_REF):
        model.train()
        total_loss = 0.0

        for images, masks in tqdm(train_loader, desc=f"Ref {epoch+1}/{EPOCHS_REF}"):
            total_loss += train_step(
                model,
                seg_loss,
                optimizer,
                images.to(device),
                masks.to(device),
            )

        model.eval()
        dice_sum = 0.0

        with torch.no_grad():
            for images, masks in val_loader:
                logits = model(images.to(device))
                dice_sum += dice_t(logits, masks.to(device), TH_REF).item()

        val_dice = dice_sum / len(val_loader)
        scheduler.step(val_dice)

        epoch_record = {
            "epoch": epoch + 1,
            "loss": total_loss / len(train_loader),
            "val_dice": val_dice,
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(epoch_record)

        print(
            f"Epoch {epoch+1} | "
            f"Loss: {epoch_record['loss']:.4f} | "
            f"Val Dice: {val_dice:.4f} | "
            f"LR: {epoch_record['lr']:.1e}"
        )

        improved = val_dice > best_dice
        if improved:
            best_dice = val_dice
            wait = 0
            torch.save(model.state_dict(), checkpoint_path("best_ref.pth"))
            print("Saved best_ref.pth")
        else:
            wait += 1

        save_checkpoint(
            checkpoint_path("last_ref.pth"),
            model,
            optimizer,
            scheduler,
            epoch + 1,
            best_dice,
            history,
        )

        if wait >= patience:
            print("Early stopping refiner")
            break

    elapsed = time.perf_counter() - start_time
    pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "refiner_history.csv"), index=False)

    metrics = {
        "best_val_dice": best_dice,
        "training_time_seconds_this_run": elapsed,
        "training_time_minutes": elapsed / 60,
    }
    save_json(metrics, os.path.join(OUT_DIR, "refiner_metrics.json"))
    return model, metrics


In [ ]:
def tta_predict(model, tensor):
    preds = []

    with torch.no_grad():
        preds.append(torch.sigmoid(model(tensor)))

        for dims in [[-1], [-2], [-1, -2]]:
            flipped = torch.flip(tensor, dims)
            preds.append(torch.flip(torch.sigmoid(model(flipped)), dims))

    return torch.stack(preds).mean(0)


# Inference and threshold search
def model_prob(model, tensor, use_tta=True):
    if use_tta:
        return tta_predict(model, tensor)

    with torch.no_grad():
        return torch.sigmoid(model(tensor))


def threshold_search(seg_model):
    print("\nThreshold Search")

    val_loader = DataLoader(
        SegDataset(val_ids),
        batch_size=1 if USE_TTA_THRESHOLD_SEARCH else BATCH,
        shuffle=False,
        **loader_kwargs,
    )

    best_th = TH_SEG
    best_dice = -1.0
    records = []

    seg_model.eval()

    for th in np.arange(0.10, 0.61, 0.02):
        dice_sum = 0.0

        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device)
                masks = masks.to(device)

                probs = model_prob(seg_model, images, use_tta=USE_TTA_THRESHOLD_SEARCH)
                pred = (probs > float(th)).float()

                inter = (pred * masks).sum()
                dice = (2.0 * inter + 1e-6) / (pred.sum() + masks.sum() + 1e-6)

                dice_sum += dice.item()

        avg_dice = dice_sum / len(val_loader)

        records.append({
            "threshold": float(th),
            "val_dice": avg_dice,
            "used_tta": USE_TTA_THRESHOLD_SEARCH,
        })

        print(f"TH={th:.2f} | Val Dice={avg_dice:.4f}")

        if avg_dice > best_dice:
            best_dice = avg_dice
            best_th = float(th)

    pd.DataFrame(records).to_csv(os.path.join(OUT_DIR, "threshold_search.csv"), index=False)

    threshold_result = {
        "best_threshold": best_th,
        "best_val_dice": best_dice,
        "used_tta": USE_TTA_THRESHOLD_SEARCH,
    }

    save_json(threshold_result, os.path.join(OUT_DIR, "best_threshold.json"))

    with open(os.path.join(OUT_DIR, "best_threshold.txt"), "w", encoding="utf-8") as f:
        f.write(f"Best threshold: {best_th}\n")
        f.write(f"Best validation Dice: {best_dice}\n")
        f.write(f"Used TTA: {USE_TTA_THRESHOLD_SEARCH}\n")

    print("Best threshold:", best_th)
    return best_th, best_dice


def predict_pipeline(image_id, cls_model, seg_model, ref_model, th_seg):
    img_f = read_f32(os.path.join(TRAIN_PATH, image_id))
    gt = build_mask(df, image_id)
    tensor = to_tensor(img_f)

    with torch.no_grad():
        cls_prob = torch.sigmoid(cls_model(tensor)).item()

    if cls_prob < TH_CLS:
        del tensor
        safe_cache()
        return img_f, gt, np.zeros((256, 1600), dtype=np.uint8), cls_prob

    prob = model_prob(seg_model, tensor, use_tta=USE_TTA_INFERENCE)[0, 0].cpu().numpy()
    coarse = (prob > th_seg).astype(np.uint8)

    roi = (prob > TH_ROI).astype(np.uint8)
    roi = cv2.dilate(
        roi,
        np.ones((DILATE_K, DILATE_K), np.uint8),
        iterations=DILATE_I,
    )

    bbox = get_bbox(roi, pad=ROI_PAD)

    if bbox is None:
        final = coarse.copy()
    else:
        x1, y1, x2, y2 = bbox

        crop = img_f[y1:y2+1, x1:x2+1]
        crop = cv2.resize(crop, (CROP_SIZE, CROP_SIZE), interpolation=cv2.INTER_LINEAR)

        crop_tensor = to_tensor(crop)

        crop_prob = model_prob(ref_model, crop_tensor, use_tta=USE_TTA_INFERENCE)[0, 0].cpu().numpy()
        crop_pred = (crop_prob > TH_REF).astype(np.uint8)

        crop_pred = cv2.resize(
            crop_pred,
            (x2 - x1 + 1, y2 - y1 + 1),
            interpolation=cv2.INTER_NEAREST,
        )

        refined = np.zeros((256, 1600), dtype=np.uint8)
        refined[y1:y2+1, x1:x2+1] = crop_pred

        # Preserve both global context and local crop refinement.
        final = np.maximum(coarse, refined)

        del crop_tensor

    final = filter_small_components(final, min_area=MIN_COMPONENT_AREA)

    final = cv2.dilate(
        final,
        np.ones((FINAL_DIL_K, FINAL_DIL_K), np.uint8),
        iterations=FINAL_DIL_I,
    )

    del tensor
    safe_cache()

    return img_f, gt, final, cls_prob


In [ ]:
# Evaluation and reporting
def evaluate(ids, cls_model, seg_model, ref_model, name="TEST", th_seg=TH_SEG):
    dice_total = 0.0
    iou_total = 0.0
    rows = []
    total_inference_time = 0.0

    for image_id in tqdm(ids, desc=name):
        start = time.perf_counter()

        _, gt, pred, cls_prob = predict_pipeline(
            image_id,
            cls_model,
            seg_model,
            ref_model,
            th_seg,
        )

        elapsed = time.perf_counter() - start
        total_inference_time += elapsed

        d = dice_np(pred, gt)
        i = iou_np(pred, gt)

        dice_total += d
        iou_total += i

        rows.append({
            "image_id": image_id,
            "gt_has_defect": int(gt.sum() > 0),
            "pred_has_defect": int(pred.sum() > 0),
            "gt_pixels": int(gt.sum()),
            "pred_pixels": int(pred.sum()),
            "dice": d,
            "iou": i,
            "classifier_probability": cls_prob,
            "inference_time_seconds": elapsed,
            "threshold_seg": th_seg,
            "threshold_cls": TH_CLS,
            "threshold_ref": TH_REF,
            "used_tta": USE_TTA_INFERENCE,
        })

    n = len(ids)
    avg_dice = dice_total / n
    avg_iou = iou_total / n
    avg_inference = total_inference_time / n

    csv_path = os.path.join(OUT_DIR, f"{name.lower()}_per_image_results.csv")
    pd.DataFrame(rows).to_csv(csv_path, index=False)

    summary = {
        "split": name,
        "num_images": n,
        "dice": avg_dice,
        "iou": avg_iou,
        "total_inference_time_seconds": total_inference_time,
        "avg_inference_time_seconds_per_image": avg_inference,
        "avg_inference_time_ms_per_image": avg_inference * 1000,
        "results_csv": csv_path,
    }

    save_json(summary, os.path.join(OUT_DIR, f"{name.lower()}_summary.json"))

    print("\n" + "=" * 45)
    print(f"{name} Dice: {avg_dice:.4f}")
    print(f"{name} IoU : {avg_iou:.4f}")
    print(f"{name} Avg inference: {avg_inference*1000:.2f} ms/image")
    print(f"Saved CSV: {csv_path}")
    print("=" * 45)

    safe_cache()
    return summary


def show_results(ids, cls_model, seg_model, ref_model, n=4, th_seg=TH_SEG, save_figures=True):
    shown = 0
    saved_paths = []

    for image_id in ids:
        if build_mask(df, image_id).sum() == 0:
            continue

        img, gt, pred, cls_prob = predict_pipeline(
            image_id,
            cls_model,
            seg_model,
            ref_model,
            th_seg,
        )

        d = dice_np(pred, gt)
        i = iou_np(pred, gt)

        fig, ax = plt.subplots(4, 1, figsize=(22, 10), dpi=130)

        ax[0].imshow(img, cmap="gray", aspect="auto")
        ax[0].set_title(f"{image_id} | Cls={cls_prob:.3f} | Dice={d:.3f} | IoU={i:.3f}")
        ax[0].axis("off")

        ax[1].imshow(gt, cmap="gray", aspect="auto")
        ax[1].set_title("Ground Truth")
        ax[1].axis("off")

        ax[2].imshow(pred, cmap="gray", aspect="auto")
        ax[2].set_title("Prediction")
        ax[2].axis("off")

        ax[3].imshow(img, cmap="gray", aspect="auto")
        ax[3].imshow(pred, cmap="autumn", alpha=0.45, aspect="auto")
        ax[3].set_title("Overlay")
        ax[3].axis("off")

        plt.tight_layout()

        if save_figures:
            safe_name = image_id.replace(".jpg", "")
            fig_path = os.path.join(VIS_DIR, f"{safe_name}_prediction.png")
            plt.savefig(fig_path, bbox_inches="tight")
            saved_paths.append(fig_path)

        plt.show()
        plt.close(fig)

        shown += 1

        if shown >= n:
            break

    save_json({"saved_visuals": saved_paths}, os.path.join(OUT_DIR, "saved_visuals.json"))
    return saved_paths


def write_final_summary(cls_metrics, seg_metrics, ref_metrics, threshold_info, val_summary, test_summary):
    total_training_minutes = (
        cls_metrics["training_time_minutes"]
        + seg_metrics["training_time_minutes"]
        + ref_metrics["training_time_minutes"]
    )

    summary = {
        "model_name": "Improved Strongest Three-Stage FPN Pipeline",
        "encoder": ENCODER_NAME,
        "architecture": {
            "stage_0": "Binary classifier",
            "stage_1": "Full-image FPN global segmentation",
            "stage_2": "Crop FPN refiner trained on Stage 1 predicted ROIs",
            "final_merge": "np.maximum(coarse, refined)",
        },
        "key_choices": [
            "Stage 1 provides global image context and ROI selection.",
            "Stage 2 refines Stage 1 regions instead of replacing them.",
            "ReduceLROnPlateau reacts to validation F1/Dice.",
            "AdamW lr=1e-4 improves stability.",
            "Low classifier threshold reduces false negatives.",
            "Connected component filtering removes tiny false positives.",
            "Resume checkpoints prevent lost training after crashes.",
        ],
        "hyperparameters": {
            "batch_size": BATCH,
            "lr": LR,
            "epochs_classifier": EPOCHS_CLS,
            "epochs_segmenter": EPOCHS_SEG,
            "epochs_refiner": EPOCHS_REF,
            "crop_size": CROP_SIZE,
            "th_cls": TH_CLS,
            "th_seg_initial": TH_SEG,
            "th_ref": TH_REF,
            "th_roi": TH_ROI,
            "roi_pad": ROI_PAD,
            "final_dilation_kernel": FINAL_DIL_K,
            "grad_clip_norm": GRAD_CLIP_NORM,
            "use_tta_inference": USE_TTA_INFERENCE,
            "use_tta_threshold_search": USE_TTA_THRESHOLD_SEARCH,
            "min_component_area": MIN_COMPONENT_AREA,
        },
        "classifier": cls_metrics,
        "segmenter": seg_metrics,
        "refiner": ref_metrics,
        "threshold_search": threshold_info,
        "validation": val_summary,
        "test": test_summary,
        "total_training_time_minutes_this_run": total_training_minutes,
    }

    json_path = os.path.join(OUT_DIR, "final_project_summary.json")
    save_json(summary, json_path)

    txt_path = os.path.join(OUT_DIR, "final_project_summary.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("Improved Strongest Three-Stage FPN Pipeline Summary\n")
        f.write("=" * 60 + "\n")
        f.write(f"Encoder: {ENCODER_NAME}\n")
        f.write(f"Best threshold: {threshold_info['best_threshold']}\n")
        f.write(f"Best threshold val Dice: {threshold_info['best_val_dice']:.4f}\n")
        f.write(f"VAL Dice: {val_summary['dice']:.4f}\n")
        f.write(f"VAL IoU: {val_summary['iou']:.4f}\n")
        f.write(f"TEST Dice: {test_summary['dice']:.4f}\n")
        f.write(f"TEST IoU: {test_summary['iou']:.4f}\n")
        f.write(f"TEST avg inference time: {test_summary['avg_inference_time_ms_per_image']:.2f} ms/image\n")
        f.write(f"Training time this run: {total_training_minutes:.2f} min\n")

    print("\nFinal summary saved:")
    print(json_path)
    print(txt_path)

    return summary


In [ ]:
if __name__ == "__main__":
    # Main execution
    total_start = time.perf_counter()

    best_cls_path = None if FORCE_RETRAIN else find_checkpoint("best_cls")
    best_seg_path = None if FORCE_RETRAIN else find_checkpoint("best_seg")
    best_ref_path = None if FORCE_RETRAIN else find_checkpoint("best_ref")

    if best_cls_path:
        cls_model = Classifier().to(device)
        load_model_weights(best_cls_path, cls_model)
        cls_metrics = skipped_metrics("classifier", best_cls_path)
    else:
        cls_model, cls_metrics = train_classifier(resume=RESUME)

    if best_seg_path:
        seg_model = make_fpn()
        load_model_weights(best_seg_path, seg_model)
        seg_metrics = skipped_metrics("segmenter", best_seg_path)
    else:
        seg_model, seg_metrics = train_segmenter(resume=RESUME)
        best_seg_path = find_checkpoint("best_seg")
        if best_seg_path:
            load_model_weights(best_seg_path, seg_model)

    seg_model.eval()

    if best_ref_path:
        ref_model = make_fpn()
        load_model_weights(best_ref_path, ref_model)
        ref_metrics = skipped_metrics("refiner", best_ref_path)
    else:
        ref_model, ref_metrics = train_refiner(seg_model, resume=RESUME)
        best_ref_path = find_checkpoint("best_ref")
        if best_ref_path:
            load_model_weights(best_ref_path, ref_model)

    cls_model.eval()
    seg_model.eval()
    ref_model.eval()

    saved_threshold = None if FORCE_RETRAIN else load_saved_threshold()
    if saved_threshold is not None:
        best_th, best_th_dice, used_threshold_tta = saved_threshold
    else:
        best_th, best_th_dice = threshold_search(seg_model)
        used_threshold_tta = USE_TTA_THRESHOLD_SEARCH

    threshold_info = {
        "best_threshold": best_th,
        "best_val_dice": best_th_dice,
        "used_tta": used_threshold_tta,
    }

    val_summary = evaluate(
        val_ids,
        cls_model,
        seg_model,
        ref_model,
        name="VAL",
        th_seg=best_th,
    )

    test_summary = evaluate(
        test_ids,
        cls_model,
        seg_model,
        ref_model,
        name="TEST",
        th_seg=best_th,
    )

    show_results(
        test_ids,
        cls_model,
        seg_model,
        ref_model,
        n=4,
        th_seg=best_th,
        save_figures=True,
    )

    write_final_summary(
        cls_metrics=cls_metrics,
        seg_metrics=seg_metrics,
        ref_metrics=ref_metrics,
        threshold_info=threshold_info,
        val_summary=val_summary,
        test_summary=test_summary,
    )

    total_elapsed = (time.perf_counter() - total_start) / 60
    print(f"\nTotal script time this run: {total_elapsed:.2f} minutes")
